# ConnectTel Telecom Customer Churn Prediction
## Notebook 04: Model Evaluation, SHAP Explainability & Artifact Saving

**Objective:**
- Evaluate candidate models on the held-out 20% test dataset.
- Generate classification reports, Confusion Matrices, and ROC Curve comparisons.
- Select best model based on empirical metric performance.
- Generate SHAP (SHapley Additive exPlanations) values to interpret key risk drivers.
- Save final trained model and preprocessing pipeline artifacts to `models/` directory.

In [ ]:
import sys
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

sys.path.append(os.path.abspath('..'))

from src.data_preprocessing import load_raw_data, clean_telco_data, get_feature_columns, build_preprocessing_pipeline
from src.feature_engineering import create_features
from src.train_models import get_baseline_models
from src.evaluation import evaluate_model, plot_confusion_matrix, plot_roc_curves, create_model_comparison_table
from src.explainability import get_feature_names_from_preprocessor, generate_shap_explanation, plot_shap_summary, plot_shap_bar

print("[INFO] Environment setup complete.")

--- 
## 1. Load Data & Prepare Test Evaluation Split

In [ ]:
raw_path = '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df_raw = load_raw_data(raw_path)
df_clean = clean_telco_data(df_raw)
df_feat = create_features(df_clean)

X = df_feat.drop(columns=['Churn', 'customerID'], errors='ignore')
y = df_feat['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

num_cols, cat_cols = get_feature_columns(X, target_col='', id_col='')
preprocessor = build_preprocessing_pipeline(num_cols, cat_cols)

--- 
## 2. Train Models & Evaluate on Test Set

In [ ]:
models_dict = get_baseline_models(random_state=42)
eval_results = []
fitted_pipelines = {}

for name, model in models_dict.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    print(f"[INFO] Fitting {name} pipeline...")
    pipeline.fit(X_train, y_train)
    fitted_pipelines[name] = pipeline
    
    res = evaluate_model(pipeline, X_test, y_test, model_name=name)
    eval_results.append(res)

--- 
## 3. Confusion Matrix & ROC Curve Comparison Plots

In [ ]:
# Plot confusion matrix for each model
for res in eval_results:
    plot_confusion_matrix(
        res['Confusion Matrix'],
        model_name=res['Model'],
        save_path=f"../outputs/figures/confusion_matrix_{res['Model'].replace(' ', '_').lower()}.png"
    )

In [ ]:
# Plot comparative ROC curves
plot_roc_curves(eval_results, y_test, save_path="../outputs/figures/roc_curves_comparison.png")

--- 
## 4. Model Comparison Table & Final Model Selection

In [ ]:
comp_table = create_model_comparison_table(eval_results)
print("=== Model Evaluation Comparison Table ===")
comp_table

In [ ]:
# Select best model (e.g. highest F1-score or ROC-AUC)
best_model_name = comp_table.iloc[0]['Model']
best_pipeline = fitted_pipelines[best_model_name]
print(f"\n[SELECTED BEST MODEL]: {best_model_name}")

--- 
## 5. Explainable AI with SHAP

In [ ]:
# Extract preprocessed feature names
fitted_prep = best_pipeline.named_steps['preprocessor']
feature_names = get_feature_names_from_preprocessor(fitted_prep)

# Transform test set for SHAP
X_test_prep = fitted_prep.transform(X_test)
classifier_model = best_pipeline.named_steps['classifier']

# Compute SHAP values
explainer, shap_values = generate_shap_explanation(classifier_model, X_test_prep, feature_names=feature_names)

In [ ]:
# SHAP Summary Plot
plot_shap_summary(shap_values, X_test_prep, feature_names=feature_names, save_path="../outputs/figures/shap_summary_plot.png")

In [ ]:
# SHAP Bar Feature Importance Plot
plot_shap_bar(shap_values, feature_names=feature_names, save_path="../outputs/figures/shap_feature_importance.png")

--- 
## 6. Save Model & Preprocessing Pipeline Artifacts

In [ ]:
os.makedirs('../models', exist_ok=True)

model_save_path = '../models/best_model.joblib'
pipeline_save_path = '../models/preprocessing_pipeline.joblib'

joblib.dump(best_pipeline, model_save_path)
joblib.dump(fitted_prep, pipeline_save_path)

print(f"[INFO] Best model saved to: {model_save_path}")
print(f"[INFO] Preprocessor saved to: {pipeline_save_path}")